# 03 — Model Training
Train the included Random Forest architecture. This is a reproducible training workflow.

In [ ]:
import pandas as pd, joblib, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

df=pd.read_csv("../data/processed/CIC-IDS2017-cleaned-combined.csv")
df.columns=df.columns.str.strip()
df.replace([float("inf"),-float("inf")],pd.NA,inplace=True)
X=df.drop(columns=["Label"]).select_dtypes(include="number")
y=df["Label"].astype(str).replace({
"Web Attack � Brute Force":"Web Attack - Brute Force",
"Web Attack � XSS":"Web Attack - XSS",
"Web Attack � Sql Injection":"Web Attack - Sql Injection"})
le=LabelEncoder(); y=le.fit_transform(y)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
prep=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])
Xt=prep.fit_transform(Xtr)
model=RandomForestClassifier(n_estimators=50,random_state=42,n_jobs=-1,class_weight="balanced_subsample",max_depth=24)
model.fit(Xt,ytr)
os.makedirs("../backend/models",exist_ok=True)
joblib.dump(model,"../backend/models/cybersecurity_model.joblib")
joblib.dump(prep,"../backend/models/preprocessor.joblib")
joblib.dump(le,"../backend/models/label_encoder.joblib")
print("Saved artifacts.")
